# ScentAI — Offline Pipeline (One-Time)

This notebook consolidates all one-time offline processing for the ScentAI project.
It is designed to run on **Google Colab with an A100 (80 GB) or Blackwell 6000 GPU**.

## What this does
1. **LLM Enrichment** — fills all empty enrichment columns (`vibe_sentence`, `formality`, `day_night`, etc.)
   for all ~35k rows in `data/vibescent_enriched.csv` using Qwen3-8B (local GPU, no API keys).
2. **Corpus Re-embedding** — embeds every row's `retrieval_text` with `Qwen3-VL-Embedding-8B`
   and saves the resulting matrix to `artifacts/qwen3vl_corpus/embeddings.npy`.

## When to run
Run this notebook **once per dataset change**. After the artifacts are committed to the repo,
the live demo notebook (`harsh_week5_qwen3vl.ipynb`) loads them directly — this notebook does not
need to run again unless the CSV changes.

## Workflow: keeping Colab in sync with local changes

1. Edit code locally → run `./sync.sh` → pushes to GitHub
2. In Colab: **run the SYNC cell** (fast git pull, no reinstall) to pick up the latest code
3. On a fresh runtime: run **Cell 1 (Environment Setup)** once, restart, then continue from Stage 2

> **Note:** Cell 1 clones the repo from GitHub directly — no zip upload required.
> Google Drive is still mounted for the `uv` package cache and output artifacts.

## Stage Map
| Stage | Cell | Description |
|-------|------|-------------|
| SYNC | SYNC cell | Pull latest code from GitHub after `./sync.sh` — no reinstall |
| 1 | Cell 1 | Environment setup — git clone, install deps. **Restart runtime after this cell.** |
| 2 | Cell 2 | Config — paths, Drive mount for outputs, HF token |
| 3 | Cell 3 | Load & inspect dataset, resume from checkpoint if available |
| 4 | Cell 4 | LLM enrichment (Qwen3-8B, vLLM native guided decoding, batch 16) |
| 5 | Cell 5 | Corpus re-embedding (Qwen3-VL-Embedding-8B, batch 64) |
| 6 | Cell 6 | Verify artifacts |
| 7 | Cell 7 | Next steps — copy outputs to repo, update settings |


In [ ]:
# SYNC — Force-pull latest code from GitHub with LFS bypass
# Run after ./sync.sh on your local machine. Skip on brand-new runtimes.
import subprocess, os, sys

REPO_DIR = '/content/vibescent'

if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    print('[INFO] Repo not yet cloned — run Stage 1 first.')
else:
    print('SYNC: Fetching latest from GitHub...')
    r = subprocess.run(
        ['git', '-C', REPO_DIR, 'fetch', 'origin', 'main', '--force'],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(f'git fetch failed: {r.stderr.strip()}')

    # Bypass LFS budget errors
    subprocess.run(['git', '-C', REPO_DIR, 'config', 'filter.lfs.smudge', 'cat'], capture_output=True)
    subprocess.run(['git', '-C', REPO_DIR, 'config', 'filter.lfs.required', 'false'], capture_output=True)

    print('SYNC: Resetting to origin/main...')
    r = subprocess.run(
        ['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(f'git reset failed: {r.stderr.strip()}')

    # Clear stale vibescents module cache so imports pick up new code
    to_delete = [m for m in sys.modules if m.startswith("vibescents")]
    for m in to_delete:
        del sys.modules[m]
    if to_delete:
        print(f'[OK] Cleared module cache: {", ".join(to_delete)}')

    commit = subprocess.run(
        ['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'],
        capture_output=True, text=True,
    ).stdout.strip()
    print(f'SYNC complete. Commit: {commit}')


In [ ]:
# Stage 1: Environment Setup
# Run ONCE on a fresh runtime → restart runtime → continue from Stage 2.
import subprocess, sys, os, shutil

REPO_DIR    = '/content/vibescent'
GITHUB_REPO = 'https://github.com/GavinGongTech/VibeScent.git'

# Step 1: Clone or sync repo
print('Step 1: Syncing repo...')
subprocess.run(['git', 'config', '--global', 'filter.lfs.smudge', 'cat'])
subprocess.run(['git', 'config', '--global', 'filter.lfs.required', 'false'])
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all'], capture_output=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'], check=True)
    print('  Repo synced to origin/main.')
else:
    if os.path.exists(REPO_DIR): shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', '--depth=1', GITHUB_REPO, REPO_DIR], check=True)
    print(f'  Cloned to {REPO_DIR}')
os.chdir(REPO_DIR)

# Step 2: Mount Drive (stores HF model cache + pipeline outputs across sessions)
print('Step 2: Mounting Google Drive...')
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    print('  Drive mounted.')
except Exception as e:
    print(f"  [WARN] Drive mount skipped: {e}")

# Step 3: Install vLLM first — its transitive deps pin the torch + transformers floor.
# MUST use pip, not uv: uv --system conflicts with Colab pre-installed CUDA torch.
print('Step 3: Installing vLLM (3-5 min on first run)...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'], check=True)
print('  vLLM done.')

# Step 4: Remaining runtime deps (torch version already locked by vLLM above)
_pkgs = [
    'google-genai', 'pandas', 'numpy', 'accelerate', 'bitsandbytes',
    'qwen-vl-utils>=0.0.14', 'json-repair', 'tqdm', 'hf_transfer',
]
print('Step 4: Installing remaining dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + _pkgs, check=True)
print('  Done.')

# Step 5: Editable project install (--no-deps avoids conflicting re-resolution)
print('Step 5: Installing vibescent project...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', REPO_DIR], check=True)
print('  Done.')

# Step 6: Upgrade transformers — Qwen3-VL-Embedding-8B requires >=4.57.0
print('Step 6: Upgrading transformers>=4.57.0...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.57.0,<5.0'], check=True)
print('  Done.')

print('\nAll packages installed.')
print('Restart runtime now (Runtime → Restart runtime), then run from Stage 2.')


In [ ]:
# Stage 2: Config
import os, sys
REPO_DIR = '/content/vibescent'
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

# Google Drive — all outputs saved here (survives session restart)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/vibescent_offline'
os.makedirs(DRIVE_BASE, exist_ok=True)

# Paths
INPUT_CSV      = os.path.join(REPO_DIR, 'data', 'vibescent_enriched.csv')
ENRICHED_CSV   = os.path.join(DRIVE_BASE, 'vibescent_enriched_full.csv')
CHECKPOINT_CSV = os.path.join(DRIVE_BASE, 'vibescent_enriched_full.csv.ckpt')
FAILURES_LOG   = os.path.join(DRIVE_BASE, 'enrichment_failures.jsonl')
EMBEDDINGS_DIR = os.path.join(DRIVE_BASE, 'qwen3vl_corpus')
os.makedirs(EMBEDDINGS_DIR, exist_ok=True)

# INPUT_CSV is gitignored — copy from Drive if not present in the cloned repo.
# First-time setup: upload vibescent_enriched.csv to one of the Drive locations listed below.
# After the pipeline completes, Stage 7 copies the enriched CSV back here automatically.
if not os.path.exists(INPUT_CSV):
    import shutil as _shutil
    _drive_fallbacks = [
        os.path.join(DRIVE_BASE, 'vibescent_enriched_full.csv'),  # enriched output from a prior run
        '/content/drive/MyDrive/vibescent_enriched.csv',           # manual upload location
    ]
    for _src in _drive_fallbacks:
        if os.path.exists(_src):
            os.makedirs(os.path.dirname(INPUT_CSV), exist_ok=True)
            _shutil.copy(_src, INPUT_CSV)
            print(f'Copied INPUT_CSV from Drive: {_src}')
            break
    else:
        print(f'[ERROR] INPUT_CSV not found: {INPUT_CSV}')
        print( '        For first-time setup, upload vibescent_enriched.csv to one of:')
        for _c in _drive_fallbacks:
            print(f'          {_c}')
        raise FileNotFoundError('Input CSV missing. See instructions above.')
else:
    _size_mb = os.path.getsize(INPUT_CSV) // (1024 * 1024)
    print(f'Input CSV found in repo: {INPUT_CSV}  ({_size_mb} MB)')

# HF model cache on Drive — prevents re-downloading on each new session
HF_CACHE = '/content/drive/MyDrive/hf_cache'
os.makedirs(HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# HF token from Colab Secrets (sidebar → 🔑 Secrets → + New secret: HF_TOKEN)
try:
    from google.colab import userdata as _ud
    _hf_token = _ud.get('HF_TOKEN') or ''
except Exception:
    _hf_token = os.environ.get('HF_TOKEN', '')

if _hf_token:
    os.environ['HF_TOKEN'] = _hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = _hf_token
    print('HF_TOKEN loaded from Colab Secrets.')
else:
    print('[WARN] HF_TOKEN not set — add via Colab sidebar → Secrets → HF_TOKEN')
    print('       Model download will fail without it.')

print(f'Config ready. Drive base: {DRIVE_BASE}')

In [ ]:
# Stage 3: Optimized Load and Inspect
import os, sys, shutil, json
import pandas as pd, numpy as np

REPO_DIR = globals().get('REPO_DIR', '/content/vibescent')
DRIVE_BASE = globals().get('DRIVE_BASE', '/content/drive/MyDrive/vibescent_offline')
INPUT_CSV = os.path.join(REPO_DIR, 'data', 'vibescent_enriched.csv')
JSONL_CACHE = os.path.join(DRIVE_BASE, 'enrichment_cache.jsonl')

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
from vibescents.enrich import ENRICHMENT_COLUMNS

df_work = pd.read_csv(INPUT_CSV)
print(f'Loaded Raw: {df_work.shape}')

# Fast Resume from JSONL Cache
if os.path.exists(JSONL_CACHE):
    print(f'Applying cache: {JSONL_CACHE}')
    cache_rows = 0
    with open(JSONL_CACHE, 'r') as f:
        for line in f:
            item = json.loads(line)
            idx = int(item['idx'])
            for col in ENRICHMENT_COLUMNS:
                if col in item: df_work.at[idx, col] = item[col]
            cache_rows += 1
    print(f'Merged {cache_rows} rows from cache.')

# Stats
done = df_work['vibe_sentence'].notna().sum() if 'vibe_sentence' in df_work.columns else 0
print(f'Status: {done}/{len(df_work)} enriched ({done/len(df_work)*100:.1f}%)')

if 'fragrance_id' not in df_work.columns:
    df_work.insert(0, 'fragrance_id', df_work.index.astype(str))
print(f'Ready. Work Shape: {df_work.shape}')

In [ ]:
# Stage 3b: Load Enrichment Model
import gc, torch, time
from vibescents.enrich import VLLMNativeEnrichmentClient

# Qwen/Qwen3-8B: confirmed working, fits on T4 (16GB) and A100 (40/80GB)
ENRICHMENT_MODEL = "Qwen/Qwen3-8B"
BATCH_SIZE = 16

if "client" in globals():
    print("Re-using existing client...")
else:
    gc.collect()
    torch.cuda.empty_cache()
    print(f"Loading {ENRICHMENT_MODEL} via vLLM...")
    print(f"VRAM before: {torch.cuda.memory_allocated()/1e9:.1f} GB")
    client = VLLMNativeEnrichmentClient(model_name=ENRICHMENT_MODEL)
    print(f"VRAM after: {torch.cuda.memory_allocated()/1e9:.1f} GB")
    print(f"Model loaded. Batch size: {BATCH_SIZE}")


In [ ]:
# Stage 4: LLM Enrichment (Qwen3-8B via vLLM native guided decoding)
import os, sys, json, traceback
import torch, gc
import pandas as pd
from tqdm.auto import tqdm

# Defensive path fallbacks
REPO_DIR       = globals().get('REPO_DIR',       '/content/vibescent')
_drive_base    = '/content/drive/MyDrive/vibescent_offline'
CHECKPOINT_CSV = globals().get('CHECKPOINT_CSV',
    os.path.join(_drive_base, 'vibescent_enriched_full.csv.ckpt'))
ENRICHED_CSV   = globals().get('ENRICHED_CSV',
    os.path.join(_drive_base, 'vibescent_enriched_full.csv'))
FAILURES_LOG   = globals().get('FAILURES_LOG',
    os.path.join(_drive_base, 'enrichment_failures.jsonl'))

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from vibescents.enrich import (
    ENRICHMENT_COLUMNS, _build_prompt, _serialize_value, build_retrieval_text
)

# df_work and client must be loaded from Stage 3 / Stage 3b
assert "df_work" in globals(), "Run Stage 3 first to load df_work."
assert "client" in globals(), "Run Stage 3b first to load the enrichment model."

_need_mask = (
    df_work["vibe_sentence"].isna()
    if "vibe_sentence" in df_work.columns
    else pd.Series([True] * len(df_work), index=df_work.index)
)
_need_idx = df_work[_need_mask].index.tolist()
print(f"Rows needing enrichment: {len(_need_idx)} / {len(df_work)}")

for col in ENRICHMENT_COLUMNS:
    if col not in df_work.columns:
        df_work[col] = None

CHECKPOINT_EVERY = 100
_completed, _failed = 0, 0
_failures = []

with tqdm(total=len(_need_idx), desc="Enriching") as pbar:
    for i in range(0, len(_need_idx), BATCH_SIZE):
        batch_idx = _need_idx[i:i+BATCH_SIZE]
        batch_prompts = [_build_prompt(df_work.loc[idx]) for idx in batch_idx]
        try:
            records = client.generate_batch(batch_prompts)
            for idx, record in zip(batch_idx, records):
                if record is None:
                    _failed += 1
                    _failures.append({"idx": idx, "error": "vLLM parse failed."})
                else:
                    for col in ENRICHMENT_COLUMNS:
                        df_work.at[idx, col] = _serialize_value(getattr(record, col))
                    _completed += 1
        except Exception as e:
            msg = f"Batch failed: {e}\n{traceback.format_exc()}"
            print(f"\n[ERROR] {msg}")
            for idx in batch_idx:
                _failed += 1
                _failures.append({"idx": idx, "error": msg})
        pbar.update(len(batch_idx))
        pbar.set_postfix(ok=_completed, fail=_failed)
        if (_completed + _failed) > 0 and (_completed + _failed) % CHECKPOINT_EVERY < BATCH_SIZE:
            df_work.to_csv(CHECKPOINT_CSV, index=False)

# Final save
df_work.to_csv(CHECKPOINT_CSV, index=False)
df_work = build_retrieval_text(df_work)
df_work.to_csv(ENRICHED_CSV, index=False)

if _failures:
    with open(FAILURES_LOG, "w") as fh:
        for r in _failures: fh.write(json.dumps(r) + "\n")

print(f"\nEnrichment done: {_completed} ok, {_failed} failed")
print(f"Saved: {ENRICHED_CSV}")

# Release vLLM weights before Stage 5 loads the embedder
try:
    client._llm = None
    del client
    import gc as _gc; _gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM after release: {torch.cuda.memory_allocated()/1e9:.1f} GB")
except Exception as _e:
    print(f"[WARN] Could not release model: {_e}")


In [ ]:
# Stage 5: Corpus Re-embedding (Qwen3-VL-Embedding-8B)
# Produces .npy embeddings in the same vector space as query-time embedding.
import sys, os, time, torch, gc, json, glob
import numpy as np
import pandas as pd
from pathlib import Path

# Defensive path fallbacks
REPO_DIR       = globals().get('REPO_DIR',       '/content/vibescent')
_drive_base    = '/content/drive/MyDrive/vibescent_offline'
ENRICHED_CSV   = globals().get('ENRICHED_CSV',   os.path.join(_drive_base, 'vibescent_enriched_full.csv'))
INPUT_CSV      = globals().get('INPUT_CSV',       os.path.join(REPO_DIR, 'data', 'vibescent_enriched.csv'))
EMBEDDINGS_DIR = globals().get('EMBEDDINGS_DIR', os.path.join(_drive_base, 'qwen3vl_corpus'))

sys.path.insert(0, os.path.join(REPO_DIR, "src"))

gc.collect(); torch.cuda.empty_cache()
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Load enriched CSV; fall back to raw input if Stage 4 output not present
_csv = ENRICHED_CSV if os.path.exists(ENRICHED_CSV) else INPUT_CSV
df_embed = pd.read_csv(_csv)
print(f"Embedding {len(df_embed)} rows from: {_csv}")

if "retrieval_text" not in df_embed.columns or df_embed["retrieval_text"].isna().all():
    from vibescents.enrich import build_retrieval_text
    df_embed = build_retrieval_text(df_embed)

texts = df_embed["retrieval_text"].fillna(df_embed.get("name", "")).tolist()
print(f"Sample: {texts[0][:120]}")

# Correct instantiation: Qwen3VLMultimodalEmbedder takes settings, not model_name
from vibescents.embeddings import Qwen3VLMultimodalEmbedder
from vibescents.settings import Settings

Qwen3VLMultimodalEmbedder._BATCH_SIZE = 64
_s = Settings()
embedder = Qwen3VLMultimodalEmbedder(settings=_s, load_in_8bit=False)
print(f"VRAM after embedder load: {torch.cuda.memory_allocated()/1e9:.1f} GB")

os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
out_emb      = os.path.join(EMBEDDINGS_DIR, "embeddings.npy")
out_manifest = os.path.join(EMBEDDINGS_DIR, "manifest.json")
CKPT_DIR     = os.path.join(EMBEDDINGS_DIR, "ckpts")
os.makedirs(CKPT_DIR, exist_ok=True)

# Resume from checkpoints
def get_checkpoints():
    return sorted(glob.glob(os.path.join(CKPT_DIR, "embeddings_ckpt_*.npy")),
                  key=lambda p: int(Path(p).stem.split("_")[-1]))

ckpt_files = get_checkpoints()
already_embedded = sum(np.load(f).shape[0] for f in ckpt_files)
print(f"Checkpoints: {len(ckpt_files)}, already embedded: {already_embedded}/{len(texts)}")

texts_to_embed = texts[already_embedded:]
if texts_to_embed:
    from tqdm.auto import tqdm
    CHUNK_SIZE = Qwen3VLMultimodalEmbedder._BATCH_SIZE * 50
    t0 = time.perf_counter()
    with tqdm(total=len(texts_to_embed), desc="Embedding") as pbar:
        for i in range(0, len(texts_to_embed), CHUNK_SIZE):
            chunk = texts_to_embed[i:i+CHUNK_SIZE]
            chunk_emb = embedder.embed_multimodal_documents(chunk)
            ckpt_path = os.path.join(CKPT_DIR, f"embeddings_ckpt_{len(get_checkpoints())}.npy")
            np.save(ckpt_path, chunk_emb)
            pbar.update(len(chunk))
    elapsed = time.perf_counter() - t0
    print(f"Embedded {len(texts_to_embed)} rows in {elapsed:.0f}s")

# Consolidate shards
all_ckpts = get_checkpoints()
embeddings = np.vstack([np.load(f) for f in all_ckpts]) if all_ckpts else np.empty((0, 0), dtype=np.float32)
print(f"Embeddings shape: {embeddings.shape}")

assert embeddings.shape[0] == len(texts), \
    f"Expected {len(texts)} embeddings, got {embeddings.shape[0]}"
assert np.isnan(embeddings).sum() == 0, "NaN values in embeddings!"
norms = np.linalg.norm(embeddings, axis=1)
assert norms.mean() > 0.99, f"Embeddings not L2-normalized (mean={norms.mean():.4f})"
print(f"Norms — mean: {norms.mean():.4f}  std: {norms.std():.5f}")

# Save numpy array (Settings.corpus_embeddings_path expects .npy)
np.save(out_emb, embeddings)
manifest = {
    "model": "Qwen/Qwen3-VL-Embedding-8B",
    "created": __import__("datetime").datetime.utcnow().isoformat(),
    "n_rows": int(embeddings.shape[0]),
    "dim": int(embeddings.shape[1]),
    "dtype": str(embeddings.dtype),
    "l2_normalized": True,
    "source_csv": _csv,
}
with open(out_manifest, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Saved: {out_emb}")
print(f"Manifest: {out_manifest}")

# Quick sanity check: top-5 most similar to row 0
sims = embeddings @ embeddings[0]
top5 = np.argsort(-sims)[1:6]
name0 = df_embed.iloc[0].get("name", "row 0")
print(f"\nTop-5 similar to \"{name0}\":" )
for idx in top5:
    print(f"  [{idx}] {df_embed.iloc[idx].get('name', f'row {idx}')} — sim={sims[idx]:.4f}")

print("\n=== All artifacts verified. Pipeline complete. ===")
print(f"Next: copy {out_emb} to artifacts/qwen3vl_corpus/ in the repo.")


# Stage 7: Next Steps — Copy Outputs to Repo

After all stages complete, copy the artifacts from Google Drive into the repo and commit:

```bash
# From repo root (or paste into a Colab shell cell with !)
mkdir -p artifacts/qwen3vl_corpus
cp /content/drive/MyDrive/vibescent_offline/qwen3vl_corpus/embeddings.npy artifacts/qwen3vl_corpus/
cp /content/drive/MyDrive/vibescent_offline/qwen3vl_corpus/manifest.json  artifacts/qwen3vl_corpus/
cp /content/drive/MyDrive/vibescent_offline/vibescent_enriched_full.csv   data/vibescent_enriched.csv
```

> **First-time setup:** `data/vibescent_enriched.csv` is gitignored and not tracked in the repo.
> Before running Cell 2, upload your raw `vibescent_enriched.csv` to Google Drive at one of:
> - `MyDrive/vibescent_offline/vibescent_enriched.csv`
> - `MyDrive/vibescent_enriched.csv`
>
> Cell 2 will copy it into the Colab session automatically.
> After the pipeline finishes, the `cp` command above writes the enriched CSV back to the repo
> so future runs can read it directly from the clone.

## `settings.py` — Already Updated

`corpus_embeddings_path` in `src/vibescents/settings.py` already points to the correct location:

```python
corpus_embeddings_path: str = str(DEFAULT_ARTIFACTS_DIR / 'qwen3vl_corpus' / 'embeddings.npy')
```

No changes to `settings.py` are needed.

## Run the Week 5 Demo Notebook Next

Open `notebooks/harsh_week5_qwen3vl.ipynb`.
**Skip Stage 0** (corpus re-embedding) — it is now complete.
Continue from Stage 1 (setup) as normal.